# Assignment 38 — CodeLlama + Ollama

Local code assistant with `codellama:7b` via Ollama + LangChain, then a Streamlit UI.

**Prereq:** Ollama running and model pulled:
```bash
ollama pull codellama:7b
ollama list
```

## PART 1 — Working with CodeLlama & Ollama

**Task 1: Setup CodeLlama with Ollama**
1. Install Ollama
2. `ollama pull codellama:7b`
3. Verify model is listed / responds

In [1]:
import subprocess

result = subprocess.run(["ollama", "list"], capture_output=True, text=True, check=False)
print(result.stdout or result.stderr)
assert "codellama" in (result.stdout or "").lower(), (
    "codellama not found — run: ollama pull codellama:7b"
)

NAME                       ID              SIZE      MODIFIED       
codellama:7b               8fdf8f752f6e    3.8 GB    16 seconds ago    
nomic-embed-text:latest    0a109f422b47    274 MB    2 days ago        



**Task 2: Basic CodeLlama Interaction**
1. Use LangChain `ChatOllama`
2. Send code prompts
3. Print responses

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = "codellama:7b"
llm = ChatOllama(model=MODEL, temperature=0.1)
print("ChatOllama ready:", MODEL)

ChatOllama ready: codellama:7b


In [3]:
res = llm.invoke("Write a Python function to check prime numbers. Keep it short.")
print(res.content)

```
def is_prime(n):
    return n > 1 and all(n % i for i in range(2, int(n ** 0.5) + 1))
```
This function takes an integer `n` as input and returns `True` if it is a prime number, and `False` otherwise. The function uses the fact that a prime number is divisible only by itself and 1. It iterates from 2 to the square root of `n`, and checks if `n` is divisible by any of these numbers. If none of them divide `n`, then it must be prime.


In [4]:
sample_code = '''
def add(a, b):
    return a + b
'''

res = llm.invoke(f"Explain this code briefly:\n{sample_code}")
print(res.content)


This is a Python function named `add` that takes two arguments `a` and `b`. The function returns the result of adding `a` and `b` together. In other words, it performs the operation `a + b`.


**Task 3 + Task 5: Code Assistant Features + Prompt Templates**

Helpers for:
1. Code generation
2. Code explanation
3. Bug fixing
4. Code optimization

Each uses a structured `ChatPromptTemplate`.

In [5]:
PROMPTS = {
    "generate": ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a senior software engineer. Generate clean, working code.\n"
            "Rules:\n"
            "- Prefer Python unless another language is requested\n"
            "- Include a short docstring\n"
            "- Output only the code in a fenced markdown block, then 2–4 bullet notes",
        ),
        ("human", "Generate code for:\n{input}"),
    ]),
    "explain": ChatPromptTemplate.from_messages([
        (
            "system",
            "You explain code clearly for a junior developer.\n"
            "Structure your answer as:\n"
            "1. What it does (1–2 sentences)\n"
            "2. Step-by-step walkthrough\n"
            "3. Edge cases / caveats",
        ),
        ("human", "Explain this code:\n{input}"),
    ]),
    "debug": ChatPromptTemplate.from_messages([
        (
            "system",
            "You debug broken code.\n"
            "Structure your answer as:\n"
            "1. Bug(s) found\n"
            "2. Why it fails\n"
            "3. Fixed code in a fenced markdown block",
        ),
        ("human", "Find and fix bugs in this code:\n{input}"),
    ]),
    "optimize": ChatPromptTemplate.from_messages([
        (
            "system",
            "You suggest practical optimizations.\n"
            "Structure your answer as:\n"
            "1. Bottlenecks / smells\n"
            "2. Suggested improvements (bullets)\n"
            "3. Optimized code in a fenced markdown block",
        ),
        ("human", "Optimize this code:\n{input}"),
    ]),
}

TASK_ALIASES = {
    "Generate Code": "generate",
    "Explain Code": "explain",
    "Debug Code": "debug",
    "Optimize Code": "optimize",
    "generate": "generate",
    "explain": "explain",
    "debug": "debug",
    "optimize": "optimize",
}


def run_task(task: str, text: str) -> str:
    key = TASK_ALIASES[task]
    chain = PROMPTS[key] | llm | StrOutputParser()
    return chain.invoke({"input": text})


def generate_code(prompt: str) -> str:
    return run_task("generate", prompt)


def explain_code(code: str) -> str:
    return run_task("explain", code)


def debug_code(code: str) -> str:
    return run_task("debug", code)


def optimize_code(code: str) -> str:
    return run_task("optimize", code)

print("helpers ready:", list(PROMPTS))

helpers ready: ['generate', 'explain', 'debug', 'optimize']


In [6]:
print("=== GENERATE ===")
print(generate_code("Write a Python function to check if a number is prime"))

=== GENERATE ===
```python
def is_prime(n):
    """
    Check if a number is prime.
    
    Args:
        n (int): The number to check.
    
    Returns:
        bool: True if the number is prime, False otherwise.
    """
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True
```
Notes:

* The function uses the Sieve of Eratosthenes algorithm to check if a number is prime.
* It starts by checking if the number is less than 2, as all numbers less than 2 are not prime.
* If the number is greater than or equal to 2, it iterates from 2 to the square root of the number and checks if any of those numbers divide the original number without leaving a remainder. If it finds a divisor, it returns False.
* If it reaches the end of the loop without finding a divisor, it returns True, indicating that the number is prime.
* The function uses the `**` operator to calculate the square root of the number and the `ran

In [7]:
print("=== EXPLAIN ===")
print(explain_code("""
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True
"""))

=== EXPLAIN ===

1. What it does: This code defines a function called `is_prime` that takes an integer `n` as input and returns whether or not it is a prime number. The function checks if the input number is less than 2, and if so, returns False. If the input number is greater than or equal to 2, the function iterates over the numbers from 2 to the square root of the input number (inclusive) and checks if any of them divide the input number evenly. If no such divisor is found, the function returns True.
2. Step-by-step walkthrough:
	* The function first checks if the input `n` is less than 2. If it is, it returns False immediately.
	* Otherwise, the function iterates over the numbers from 2 to the square root of the input number (inclusive) using a for loop.
	* Inside the for loop, the function checks if the current number `i` divides the input number `n` evenly. If it does, the function returns False immediately.
	* If the for loop completes without finding any divisors, the function 

In [8]:
print("=== DEBUG ===")
print(debug_code("""
def average(nums):
    total = 0
    for n in nums:
        total += n
    return total / len(nums)  # crashes on empty list
"""))

=== DEBUG ===

1. Bug found: The code crashes when the input list is empty.
2. Why it fails: When the input list is empty, the `len` function returns 0, and then the division by zero occurs in the return statement. This causes a ZeroDivisionError to be raised.
3. Fixed code in a fenced markdown block:
```
def average(nums):
    if len(nums) == 0:
        return None
    total = 0
    for n in nums:
        total += n
    return total / len(nums)
```


In [9]:
print("=== OPTIMIZE ===")
print(optimize_code("""
def unique(items):
    out = []
    for x in items:
        if x not in out:
            out.append(x)
    return out
"""))

=== OPTIMIZE ===

1. Bottlenecks / smells:
	* The `for` loop is the main bottleneck, as it iterates over each item in the input list and checks if it's already in the output list. This can be slow for large inputs.
	* The `if` statement inside the loop also has a time complexity of O(n), where n is the length of the input list. This means that the overall time complexity of the function is O(n^2).
2. Suggested improvements:
	* Use a set to keep track of unique items instead of a list. This will have a time complexity of O(1) for lookups, making the function much faster for large inputs.
	* Use the `extend` method instead of appending to the output list in a loop. This will also make the function faster, as it reduces the number of times the list needs to be resized.
3. Optimized code:
```
def unique(items):
    out = set()
    for x in items:
        if x not in out:
            out.add(x)
    return list(out)
```


## PART 2 — Streamlit app

**Task 4:** run the app after the model is pulled:
```bash
cd genai/langchain/assignment-38
streamlit run app.py
```

UI: text area + task dropdown (Generate / Explain / Debug / Optimize) + CodeLlama output.